In [2]:
import os
import glob
import datetime
import numpy as np
import pandas as pd
import xarray as xr

In [3]:
### automatically refresh the buffer
%load_ext autoreload
%autoreload 2

### solve the auto-complete issue

%config Completer.use_jedi = False
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

### lvl 2 setups (systerm)
import os
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap,LinearSegmentedColormap,BoundaryNorm
import matplotlib.dates as mdates
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime


In [11]:
# 输出目录
out_dir = "/data/ggong/ARM_monthly/ENA/interpolatedsondeC1"
os.makedirs(out_dir, exist_ok=True)

months = pd.date_range("2016-01-01", "2023-12-31", freq="MS")

### test

In [12]:
ds_mete = xr.open_mfdataset('/data/shared_data/ARM_data/ENA/others/interpolatedsondeC1.c1/enainterpolatedsondeC1.c1.202312*.nc')

ds_mete_sel = ds_mete.sel(height=slice(0.0, 8.1))

vars_to_save = [
    "rh", "qc_rh",
    "u_wind", "qc_u_wind",
    "temp", "qc_temp",
    "potential_temp", "qc_potential_temp",
    
]

ds_mete_vars = ds_mete_sel[vars_to_save]

In [13]:
ds = ds_mete_vars

### Covariation of meteorological factors calculate

In [42]:
import xarray as xr
import numpy as np

def compute_RH_TEMP_VWS_LTS(ds):
    """
    输入
    ----
    ds : xarray.Dataset
        需要包含变量（dims: time, height）及其 QC：
          rh, qc_rh,
          temp, qc_temp,
          u_wind, qc_u_wind,
          potential_temp, qc_potential_temp
        并且具有 height 坐标（单位 km，升序/降序均可）

    输出
    ----
    out : xarray.Dataset
        含 4 个时间序列变量：
          RH_750_850      (%),  1.4567–2.4652 km 的高度加权平均（nearest 到网格）
          TEMP_750_850    (K),  同上
          VWS_725_925     (m s-1) = |u(2.7343 km) - u(0.7617 km)|
          LTS             (K) = θ(3.0109 km) - θ(surface = height 索引 0)
    """
    # ---- 校验 ----
    need = [
        "rh","qc_rh","temp","qc_temp",
        "u_wind","qc_u_wind","potential_temp","qc_potential_temp"
    ]
    missing = [n for n in need if n not in ds]
    if missing:
        raise KeyError(f"Dataset 缺少变量: {missing}")
    if "height" not in ds.dims:
        raise KeyError("Dataset 需包含维度 'height'（单位 km）")

    # ---- 确保 height 升序 ----
    ds = ds.sortby("height")

    # ---- QC 过滤（仅保留 qc==0）----
    rh     = ds["rh"             ].where(ds["qc_rh"]             == 0)
    T      = ds["temp"           ].where(ds["qc_temp"]           == 0)
    u      = ds["u_wind"         ].where(ds["qc_u_wind"]         == 0)
    theta  = ds["potential_temp" ].where(ds["qc_potential_temp"] == 0)

    # ---- 工具：将目标高度映射到最近网格高度（返回 float）----
    def nearest_height(val_km: float) -> float:
        # sel(..., method='nearest') 返回 DataArray，取其坐标的标量值
        h = ds["height"].sel(height=val_km, method="nearest").item()
        return float(h)

    # ---- 高度加权平均工具 (∫v dh / ∫dh) ----
    def layer_mean_hweighted(da, hmin, hmax):
        """
        da: DataArray(time, height)
        返回：DataArray(time,)
        做法：对 [hmin, hmax] 内的层，使用厚度权重（基于 height 的梯度）
        """
        # 将端点对齐到最近网格高度
        h0 = nearest_height(hmin)
        h1 = nearest_height(hmax)
        if h0 > h1:
            h0, h1 = h1, h0

        sub = da.sel(height=slice(h0, h1))
        # 若区间内为空，直接返回全 NaN
        if sub.sizes.get("height", 0) == 0:
            return xr.full_like(da.isel(height=0), np.nan).drop_vars("height")

        # 以原始网格的几何厚度作为权重
        h = sub["height"]
        # np.gradient 需要 1D array
        dh = np.gradient(h.values)
        w = xr.DataArray(dh, coords={"height": h}, dims=("height",))

        # 有效值的权重和，避免 NaN 传播
        num = (sub * w).sum(dim="height", skipna=True)
        den = w.where(sub.notnull()).sum(dim="height", skipna=True)

        out = (num / den)
        return out

    # ---- 1) RH/T 在 1.4567–2.4652 km 的高度加权平均 ----
    hmin, hmax = 1.4567, 2.4652
    RH_750_850   = layer_mean_hweighted(rh, hmin, hmax).rename("RH_750_850")
    RH_750_850.attrs.update({"units":"%", "long_name":"Height-weighted RH (≈750–850 hPa)"})

    TEMP_750_850 = layer_mean_hweighted(T,  hmin, hmax).rename("TEMP_750_850")
    # 温度通常是 K，这里按数据原单位，不强行改单位
    TEMP_750_850.attrs.update({"units": T.attrs.get("units","K"),
                               "long_name":"Height-weighted Temperature (≈750–850 hPa)"})

    # ---- 2) VWS：|u(2.7343 km) - u(0.7617 km)|（m/s) ----
    h_725  = nearest_height(2.7343)  # ≈725 hPa
    h_925  = nearest_height(0.7617)  # ≈925 hPa
    u_725  = u.sel(height=h_725, method="nearest")
    u_925  = u.sel(height=h_925, method="nearest")
    VWS_725_925 = (u_725 - u_925).rename("VWS_725_925")
    VWS_725_925.attrs.update({"units":"m s-1",
                              "long_name":"Zonal wind shear |u(≈725 hPa) - u(≈925 hPa)|"})

    # ---- 3) LTS = θ(3.0109 km) - θ(surface(height index==0)) ----
    h_700 = nearest_height(3.0109)  # ≈700 hPa
    theta_700     = theta.sel(height=h_700, method="nearest")
    theta_surface = theta.isel(height=0)
    LTS = (theta_700 - theta_surface).rename("LTS")
    LTS.attrs.update({"units": theta.attrs.get("units","K"),
                      "long_name":"Lower Tropospheric Stability (θ≈700 hPa − θ at surface)"})

    # ---- 合并输出 ----
    out = xr.merge([RH_750_850, TEMP_750_850, VWS_725_925, LTS])
    out.attrs["note"] = ("QC==0 used. RH/T are height-weighted means over ~750–850 hPa "
                         "(endpoints snapped to nearest height). VWS=|u(≈725)-u(≈925)|. "
                         "LTS=θ(≈700)-θ(surface at height index 0).")
    return out


In [43]:
res = compute_RH_TEMP_VWS_LTS(ds)

In [47]:
# ===== 路径与参数 =====
base_dir = "/data/shared_data/ARM_data/ENA/others/interpolatedsondeC1.c1"
out_dir  = "/data/ggong/ARM_monthly/ENA/interpolatedsondeC1"
os.makedirs(out_dir, exist_ok=True)

vars_to_save = [
    "rh", "qc_rh",
    "u_wind", "qc_u_wind",
    "temp", "qc_temp",
    "potential_temp", "qc_potential_temp",
]

# ===== 月度序列（可按需修改时间范围）=====
months = pd.date_range("2016-01-01", "2023-12-01", freq="MS")



def _preprocess(ds):
    # 只保留存在于 ds 的变量，避免某些月份缺列导致报错
    keep = [v for v in vars_to_save if v in ds.variables]
    # 也保留坐标与必要辅助变量
    keep_coords = []
    for c in ["time", "height"]:
        if c in ds.coords or c in ds.variables:
            keep_coords.append(c)
    ds = ds[keep + keep_coords]

    return ds



# ===== 月度循环处理 =====
for m in months:
    ym = m.strftime("%Y%m")
    pattern = os.path.join(base_dir, f"enainterpolatedsondeC1.c1.{ym}*.nc")
    files = sorted(glob.glob(pattern))

    if not files:
        print(f"[SKIP] {ym}: 未找到文件")
        continue

    try:
        # 读入该月所有文件（自动按坐标对齐），仅取需要变量
        ds = xr.open_mfdataset(
            files,
            combine="by_coords",
            preprocess=_preprocess,
            parallel=True,
            decode_times=True,
            engine=None  # 让 xarray 自动判断引擎
        )

        # 计算 4 个指标（函数需已定义好）
        ds_out = compute_RH_TEMP_VWS_LTS(ds)

        # 输出文件名与路径
        out_name = f"interpolatedsondeC1_ENA_{ym}_height_average.nc"
        out_path = os.path.join(out_dir, out_name)

        # 不设置压缩，直接保存
        ds_out.to_netcdf(out_path)

        print(f"[OK] {ym}: output {out_path}")
        print(datetime.datetime.now())

    except Exception as e:
        print(f"[ERROR] {ym}: {e}")

    finally:
        # 及时关闭文件，释放资源
        try:
            ds.close()
        except Exception:
            pass
        try:
            ds_out.close()
        except Exception:
            pass


[OK] 201601: output /data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_201601_height_average.nc
2025-10-29 12:46:22.150801
[OK] 201602: output /data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_201602_height_average.nc
2025-10-29 12:47:44.030702
[OK] 201603: output /data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_201603_height_average.nc
2025-10-29 12:49:18.537569
[OK] 201604: output /data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_201604_height_average.nc
2025-10-29 12:50:35.586473
[OK] 201605: output /data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_201605_height_average.nc
2025-10-29 12:51:53.477070
[OK] 201606: output /data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_201606_height_average.nc
2025-10-29 12:53:04.315241
[OK] 201607: output /data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_201607_height_average.nc
2025-10-29 12:54:31

In [51]:
ds_1 = xr.open_mfdataset('/data/ggong/ARM_monthly/ENA/interpolatedsondeC1/*')

In [54]:
START = "2016-01-01 00:00:00"
END   = "2023-12-31 23:59:00"

# 确保时间升序
ds_1 = ds_1.sortby("time")

# 目标 2 分钟时间轴
target_time = pd.date_range(START, END, freq="2min")

# 沿时间维线性插值到 2 分钟分辨率
ds_2min = ds_1.interp(time=target_time, method="linear")

# （可选）检查结果
print(ds_2min)
print(f"新时间点数量: {len(ds_2min.time)}")

<xarray.Dataset> Size: 50MB
Dimensions:       (time: 2103840)
Coordinates:
  * time          (time) datetime64[ns] 17MB 2016-01-01 ... 2023-12-31T23:58:00
Data variables:
    RH_750_850    (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
    TEMP_750_850  (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
    VWS_725_925   (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
    LTS           (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
Attributes:
    units:      %
    long_name:  Height-weighted RH (≈750–850 hPa)
    note:       QC==0 used. RH/T are height-weighted means over ~750–850 hPa ...
新时间点数量: 2103840


In [55]:
ds_2min

<xarray.Dataset> Size: 50MB
Dimensions:       (time: 2103840)
Coordinates:
  * time          (time) datetime64[ns] 17MB 2016-01-01 ... 2023-12-31T23:58:00
Data variables:
    RH_750_850    (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
    TEMP_750_850  (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
    VWS_725_925   (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
    LTS           (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
Attributes:
    units:      %
    long_name:  Height-weighted RH (≈750–850 hPa)
    note:       QC==0 used. RH/T are height-weighted means over ~750–850 hPa ...

In [56]:
ds_2min.to_netcdf('/data/ggong/ARM_monthly/ENA/RH_T_VWS_LTS_2min.nc')

## calculate mete2 

In [14]:
import xarray as xr
import numpy as np

def compute_SurT_SurWind_RH34_T34(ds):
    """
    输入
    ----
    ds : xarray.Dataset
        需要包含变量（dims: time, height）及其 QC：
          temp, qc_temp,
          u_wind, qc_u_wind,
          v_wind, qc_v_wind,
          rh,   qc_rh
        并且具有 height 坐标（单位 km，升序/降序均可）

    输出
    ----
    out : xarray.Dataset
        含 4 个时间序列变量：
          T_surf    (℃)     地表温度，对应于 ds.height[0]
          T_34      (℃)     3–4 km 的高度 temp 的几何厚度加权平均
          Wind_surf (m s-1) 地表风速，基于 u/v 计算
          RH_34     (%)     3–4 km 的高度 rh 的几何厚度加权平均
    """
    # ---- 校验 ----
    need = [
        "rh", "qc_rh",
        "temp", "qc_temp",
        "u_wind", "qc_u_wind",
        "v_wind", "qc_v_wind",
    ]
    missing = [n for n in need if n not in ds]
    if missing:
        raise KeyError(f"Dataset 缺少变量: {missing}")
    if "height" not in ds.dims:
        raise KeyError("Dataset 需包含维度 'height'（单位 km）")

    # ---- 确保 height 升序 ----
    ds = ds.sortby("height")

    # ---- QC 过滤（仅保留 qc==0）----
    rh = ds["rh"].where(ds["qc_rh"] == 0)
    T  = ds["temp"].where(ds["qc_temp"] == 0)
    u  = ds["u_wind"].where(ds["qc_u_wind"] == 0)
    v  = ds["v_wind"].where(ds["qc_v_wind"] == 0)

    # ---- 工具：将目标高度映射到最近网格高度（返回 float）----
    def nearest_height(val_km: float) -> float:
        # sel(..., method='nearest') 返回 DataArray，取其坐标的标量值
        h = ds["height"].sel(height=val_km, method="nearest").item()
        return float(h)

    # ---- 高度加权平均工具 (∫v dh / ∫dh) ----
    def layer_mean_hweighted(da, hmin, hmax):
        """
        da: DataArray(time, height)
        返回：DataArray(time,)
        做法：对 [hmin, hmax] 内的层，使用厚度权重（基于 height 的梯度）
        """
        # 将端点对齐到最近网格高度
        h0 = nearest_height(hmin)
        h1 = nearest_height(hmax)
        if h0 > h1:
            h0, h1 = h1, h0

        sub = da.sel(height=slice(h0, h1))

        # 若区间内为空，直接返回全 NaN（time 维度）
        if sub.sizes.get("height", 0) == 0:
            # da.isel(height=0) -> (time,) 的 DataArray
            return xr.full_like(da.isel(height=0), np.nan)

        # 以原始网格的几何厚度作为权重
        h = sub["height"]
        dh = np.gradient(h.values)  # 1D 数组
        w = xr.DataArray(dh, coords={"height": h}, dims=("height",))

        # 有效值的权重和，避免 NaN 传播
        num = (sub * w).sum(dim="height", skipna=True)
        den = w.where(sub.notnull()).sum(dim="height", skipna=True)

        out = num / den
        return out

    # ---- 1) RH/T 在 3-4 km 的高度加权平均 ----
    hmin, hmax = 3, 4
    RH_34 = layer_mean_hweighted(rh, hmin, hmax).rename("RH_34")
    RH_34.attrs.update({
        "units": "percent",
        "long_name": "Height-weighted relative humidity (~3–4 km)",
        "layer_bounds_km": [hmin, hmax],
    })

    T_34 = layer_mean_hweighted(T, hmin, hmax).rename("T_34")
    T_34.attrs.update({
        "units": T.attrs.get("units", "degC"),
        "long_name": "Height-weighted temperature (~3–4 km)",
        "layer_bounds_km": [hmin, hmax],
    })

    # ---- 2) Surface Wind（m/s), Surface T(℃) ----
    # 这里按照 “height[0] 是最低层（近地面）” 的约定来取
    T_surf = T.isel(height=0).rename("T_surf")
    T_surf.attrs.update({
        "units": T.attrs.get("units", "degC"),
        "long_name": "Surface temperature (at lowest height level)",
        "note": "QC filtered: qc_temp == 0",
        "height_level_km": float(ds["height"].isel(height=0)),
    })

    # u、v 已经过 QC 过滤
    Wind_surf = np.hypot(
        u.isel(height=0),
        v.isel(height=0)
    ).rename("Wind_surf")
    Wind_surf.attrs.update({
        "units": "m s-1",
        "long_name": "Surface wind speed (from u, v at lowest height level)",
        "note": "QC filtered: qc_u_wind == 0 & qc_v_wind == 0",
        "height_level_km": float(ds["height"].isel(height=0)),
    })

    # ---- 合并输出 ----
    out = xr.merge([RH_34, T_34, Wind_surf, T_surf])
    out.attrs["note"] = (
        "Derived time series from profile dataset: "
        "QC filtering (qc_* == 0) applied to temp, rh, u_wind, v_wind. "
        "T_34 and RH_34 are geometrically height-weighted means over ~3–4 km, "
        "using layer thickness (Δheight) as weights. "
        "T_surf and Wind_surf are taken at the lowest height level."
    )

    return out


In [16]:
# ===== 路径与参数 =====
base_dir = "/data/shared_data/ARM_data/SGP/others/sgpinterpolatedsondeC1.c1"
out_dir  = "/data/ggong/ARM_monthly/SGP/interpolatedsondeC1"
os.makedirs(out_dir, exist_ok=True)

vars_to_save = [
        "rh", "qc_rh",
        "temp", "qc_temp",
        "u_wind", "qc_u_wind",
        "v_wind", "qc_v_wind",
    ]
    

# ===== 月度序列（可按需修改时间范围）=====
months = pd.date_range("2016-01-01", "2023-12-01", freq="MS")



def _preprocess(ds):
    # 只保留存在于 ds 的变量，避免某些月份缺列导致报错
    keep = [v for v in vars_to_save if v in ds.variables]
    # 也保留坐标与必要辅助变量
    keep_coords = []
    for c in ["time", "height"]:
        if c in ds.coords or c in ds.variables:
            keep_coords.append(c)
    ds = ds[keep + keep_coords]

    return ds



# ===== 月度循环处理 =====
for m in months:
    ym = m.strftime("%Y%m")
    pattern = os.path.join(base_dir, f"sgpinterpolatedsondeC1.c1.{ym}*.nc")
    files = sorted(glob.glob(pattern))

    if not files:
        print(f"[SKIP] {ym}: 未找到文件")
        continue

    try:
        # 读入该月所有文件（自动按坐标对齐），仅取需要变量
        ds = xr.open_mfdataset(
            files,
            combine="by_coords",
            preprocess=_preprocess,
            parallel=True,
            decode_times=True,  
        )

        # 计算 4 个指标（函数需已定义好）
        ds_out = compute_SurT_SurWind_RH34_T34(ds)

        # 输出文件名与路径
        out_name = f"interpolatedsondeC1_SGP_{ym}_height_average_mete2.nc"
        out_path = os.path.join(out_dir, out_name)

        # 不设置压缩，直接保存
        ds_out.to_netcdf(out_path)

        print(f"[OK] {ym}: output {out_path}")
        print(datetime.datetime.now())

    except Exception as e:
        print(f"[ERROR] {ym}: {e}")

    finally:
        # 及时关闭文件，释放资源
        try:
            ds.close()
        except Exception:
            pass
        try:
            ds_out.close()
        except Exception:
            pass

[OK] 201601: output /data/ggong/ARM_monthly/SGP/interpolatedsondeC1/interpolatedsondeC1_SGP_201601_height_average_mete2.nc
[ERROR] 201601: type object 'datetime.datetime' has no attribute 'datetime'
[OK] 201602: output /data/ggong/ARM_monthly/SGP/interpolatedsondeC1/interpolatedsondeC1_SGP_201602_height_average_mete2.nc
[ERROR] 201602: type object 'datetime.datetime' has no attribute 'datetime'
[OK] 201603: output /data/ggong/ARM_monthly/SGP/interpolatedsondeC1/interpolatedsondeC1_SGP_201603_height_average_mete2.nc
[ERROR] 201603: type object 'datetime.datetime' has no attribute 'datetime'
[OK] 201604: output /data/ggong/ARM_monthly/SGP/interpolatedsondeC1/interpolatedsondeC1_SGP_201604_height_average_mete2.nc
[ERROR] 201604: type object 'datetime.datetime' has no attribute 'datetime'
[OK] 201605: output /data/ggong/ARM_monthly/SGP/interpolatedsondeC1/interpolatedsondeC1_SGP_201605_height_average_mete2.nc
[ERROR] 201605: type object 'datetime.datetime' has no attribute 'datetime'
[OK] 

In [23]:
ds_2 = xr.open_mfdataset('/data/ggong/ARM_monthly/SGP/interpolatedsondeC1/*mete2*')

In [24]:
ds_2

<xarray.Dataset> Size: 100MB
Dimensions:    (time: 4161600)
Coordinates:
  * time       (time) datetime64[ns] 33MB 2016-01-01T00:00:30 ... 2023-12-31T...
    height     float32 4B 0.318
Data variables:
    RH_34      (time) float32 17MB dask.array<chunksize=(44640,), meta=np.ndarray>
    T_34       (time) float32 17MB dask.array<chunksize=(44640,), meta=np.ndarray>
    Wind_surf  (time) float32 17MB dask.array<chunksize=(44640,), meta=np.ndarray>
    T_surf     (time) float32 17MB dask.array<chunksize=(44640,), meta=np.ndarray>
Attributes:
    units:            percent
    long_name:        Height-weighted relative humidity (~3–4 km)
    layer_bounds_km:  [3 4]
    note:             Derived time series from profile dataset: QC filtering ...

In [18]:
START = "2016-01-01 00:00:00"
END   = "2023-12-31 23:59:00"

# 确保时间升序
ds_2 = ds_2.sortby("time")

# 目标 2 分钟时间轴
target_time = pd.date_range(START, END, freq="2min")

# 沿时间维线性插值到 2 分钟分辨率
ds_2min = ds_2.interp(time=target_time, method="linear")

# （可选）检查结果
print(ds_2min)
print(f"新时间点数量: {len(ds_2min.time)}")

<xarray.Dataset> Size: 50MB
Dimensions:    (time: 2103840)
Coordinates:
    height     float32 4B 0.318
  * time       (time) datetime64[ns] 17MB 2016-01-01 ... 2023-12-31T23:58:00
Data variables:
    RH_34      (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
    T_34       (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
    Wind_surf  (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
    T_surf     (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
Attributes:
    units:            percent
    long_name:        Height-weighted relative humidity (~3–4 km)
    layer_bounds_km:  [3 4]
    note:             Derived time series from profile dataset: QC filtering ...
新时间点数量: 2103840


In [4]:
# ds_2min.to_netcdf('/data/ggong/ARM_monthly/SGP/RH34_T34_Tsurf_Windsurf_2min.nc')

## LTS = Cloud height + 1 resolution 

In [3]:
import xarray as xr
import pandas as pd

START = "2016-01-01 00:00:00"
END   = "2023-12-31 23:59:00"

ds_mete = xr.open_mfdataset(
    '/data/shared_data/ARM_data/SGP/others/sgpinterpolatedsondeC1.c1/sgpinterpolatedsondeC1.c1.*.nc',
    chunks={"time": 1000},   # 显式给个还不错的时间 chunk
)

KeyboardInterrupt: 

In [73]:
# ds_mete_sel = ds_mete.sel(height=slice(0.0, 8.1))

# valid = ds_mete_sel["qc_potential_temp"] == 0
# potential_temp_qc = ds_mete_sel["potential_temp"].where(valid)

# target_time = pd.date_range(START, END, freq="2min")

# ds_2min = potential_temp_qc.interp(time=target_time, method="linear")

# # 写出时也是按 chunk 流式写
# ds_2min.to_netcdf("/data/ggong/ARM_monthly/SGP/potential_temp_SGP_2min.nc")


In [13]:
ds_lts = xr.open_dataset('/data/ggong/ARM_monthly/ENA/potential_temp_ENA_2min.nc')

In [14]:
ds_ctop = xr.open_dataset('/data/ggong/ARM_monthly/ENA/cloud_height.nc')

In [15]:
c_top = ds_ctop.cloud_top_height.values

In [16]:
import xarray as xr
import numpy as np

def compute_cloud_top_LTS(ds_lts, ds_ctop,
                          theta_name="potential_temp",
                          ctop_name="cloud_top_height"):
    """
    计算 LTS = θ(cloud top 上一层) - θ(surface, height 索引 0)

    参数
    ----
    ds_lts : xarray.Dataset
        包含 (time, height) 的位势温度字段，例如：
        - potential_temp(time, height)
        - height(height)

    ds_ctop : xarray.Dataset
        包含云顶高度时间序列：
        - cloud_top_height(time)

    返回
    ----
    LTS : xarray.DataArray  (time,)
    """

    # ---- 合并两个 dataset，保证 time 对齐 ----
    ds = xr.merge(
        [ds_lts[[theta_name]], ds_ctop[[ctop_name]]],
        join="inner"
    )

    # ---- 确保 height 升序 ----
    ds = ds.sortby("height")

    theta = ds[theta_name]          # (time, height)
    ctop  = ds[ctop_name]           # (time,)

    # ---- 广播出 (time, height) 的高度 & 云顶高度 ----
    H, C = xr.broadcast(ds["height"], ctop)    # H,C: (time, height)

    # mask: higher than cloud top
    mask = H > C                               # (time, height)

    # 只保留“第一个 True”那一层：cumsum==1 的位置
    first_mask = mask & (mask.cumsum("height") == 1)

    # 在这些网格点上取 θ。其它高度都是 NaN。
    theta_first_above = theta.where(first_mask)

    # 每个时间只剩一个有效高度，沿 height 取最大值即可拿到这一层 θ
    theta_ctop_plus = theta_first_above.max("height", skipna=True)

    # 近地面 θ：height 索引 0
    theta_surface = theta.isel(height=0)

    LTS = (theta_ctop_plus - theta_surface).rename("LTS")
    LTS.attrs.update({
        "units": theta.attrs.get("units", "K"),
        "long_name": "Lower Tropospheric Stability (theta at first level above cloud top minus surface)"
    })

    return LTS


In [90]:
LTS = compute_cloud_top_LTS(ds_lts, ds_ctop)

In [19]:
# LTS.to_netcdf('/data/ggong/ARM_monthly/SGP/LTS_cloud_top_plus.nc')

## LTS = Cloud height + 2 resolution 

In [17]:
import xarray as xr
import numpy as np

def compute_cloud_top_LTS(ds_lts, ds_ctop,
                          theta_name="potential_temp",
                          ctop_name="cloud_top_height"):
    """
    计算 LTS = θ(第一层高于 cloud_top + 0.4 km 的高度) - θ(surface, height 索引 0)

    参数
    ----
    ds_lts : xarray.Dataset
        包含 (time, height) 的位势温度字段，例如：
        - potential_temp(time, height)
        - height(height)   # 单位：km

    ds_ctop : xarray.Dataset
        包含云顶高度时间序列：
        - cloud_top_height(time)  # 单位：km

    返回
    ----
    LTS : xarray.DataArray  (time,)
    """

    # ---- 合并两个 dataset，保证 time 对齐 ----
    ds = xr.merge(
        [ds_lts[[theta_name]], ds_ctop[[ctop_name]]],
        join="inner"
    )

    # ---- 确保 height 升序 ----
    ds = ds.sortby("height")

    theta = ds[theta_name]      # (time, height)
    ctop  = ds[ctop_name]       # (time,)

    # ===== 关键修改：阈值改为 cloud_top + 0.4 km =====
    ctop_plus = ctop + 0.4      # (time,)

    # 广播出 (time, height) 的高度 & 云顶+0.4km 高度
    H, C = xr.broadcast(ds["height"], ctop_plus)   # H, C: (time, height)

    # mask: 高于 cloud_top+0.4km 的所有格点
    mask = H > C                                # (time, height)

    # 只保留“第一个 True”那一层：cumsum==1 的位置
    first_mask = mask & (mask.cumsum("height") == 1)

    # 在这些网格点上取 θ。其它高度都是 NaN。
    theta_first_above = theta.where(first_mask)

    # 每个时间只剩一个有效高度，沿 height 取最大值即可拿到这一层 θ
    theta_ctop_plus = theta_first_above.max("height", skipna=True)

    # 近地面 θ：height 索引 0
    theta_surface = theta.isel(height=0)

    LTS = (theta_ctop_plus - theta_surface).rename("LTS")
    LTS.attrs.update({
        "units": theta.attrs.get("units", "K"),
        "long_name": "Lower Tropospheric Stability (theta at first level above cloud top+0.4 km minus surface)"
    })

    return LTS


In [18]:
LTS = compute_cloud_top_LTS(ds_lts, ds_ctop)

In [20]:
LTS.to_netcdf('/data/ggong/ARM_monthly/ENA/LTS_cloud_top_plus_2.nc')

## SPG

In [57]:
ds_1 = xr.open_mfdataset('/data/ggong/ARM_monthly/SGP/interpolatedsondeC1/*')

In [58]:
START = "2016-01-01 00:00:00"
END   = "2023-12-31 23:59:00"

# 确保时间升序
ds_1 = ds_1.sortby("time")

# 目标 2 分钟时间轴
target_time = pd.date_range(START, END, freq="2min")

# 沿时间维线性插值到 2 分钟分辨率
ds_2min = ds_1.interp(time=target_time, method="linear")

# （可选）检查结果
print(ds_2min)
print(f"新时间点数量: {len(ds_2min.time)}")

<xarray.Dataset> Size: 50MB
Dimensions:       (time: 2103840)
Coordinates:
  * time          (time) datetime64[ns] 17MB 2016-01-01 ... 2023-12-31T23:58:00
Data variables:
    RH_750_850    (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
    TEMP_750_850  (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
    VWS_725_925   (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
    LTS           (time) float32 8MB dask.array<chunksize=(2103840,), meta=np.ndarray>
Attributes:
    units:      %
    long_name:  Height-weighted RH (≈750–850 hPa)
    note:       QC==0 used. RH/T are height-weighted means over ~750–850 hPa ...
新时间点数量: 2103840


In [60]:
ds_2min.to_netcdf('/data/ggong/ARM_monthly/SGP/RH_T_VWS_LTS_2min.nc')

## precip

In [1]:
import cdflib
import xarray as xr
import numpy as np

In [24]:
ds = xr.open_mfdataset("/data/shared_data/ARM_data/SGP/others/sgpvdisC1.b1/sgp*.cdf")

In [25]:
qc_name = "qc_rain_rate" if "qc_rain_rate" in ds.variables else "qs_rain_rate"

# 2) 应用 QC（仅保留 qc==0）
rr_qc = ds["rain_rate"].where(ds[qc_name] == 0)

# 3) 补齐完整 1 分钟时间轴
t1min = pd.date_range("2016-01-01 00:00:00", "2023-12-31 23:59:00", freq="1min")
rr_qc_1min = rr_qc.reindex(time=t1min)

In [27]:
rr_qc_2min = rr_qc_1min.isel(time=slice(0, None, 2))

In [31]:
rr_qc_2min.to_netcdf('/data/ggong/ARM_monthly/SGP/vdis_qc_rain_rate.nc')

## U_ V_ Wind Speed

In [1]:
import os
import xarray as xr
import pandas as pd
from pathlib import Path

base_dir = "/data/shared_data/ARM_data/ENA/others/interpolatedsondeC1.c1"
out_dir  = "/data/ggong/ARM_monthly/ENA/UV_wind_rose"
Path(out_dir).mkdir(parents=True, exist_ok=True)

target_heights = [0.030, 1.456, 2.465]
vars_to_save = ["u_wind", "qc_u_wind", "v_wind", "qc_v_wind"]

months = pd.period_range("2016-01", "2023-12", freq="M")

for p in months:
    yyyymm = p.strftime("%Y%m")
    pattern = f"{base_dir}/enainterpolatedsondeC1.c1.{yyyymm}*.nc"
    files = sorted(list(Path(base_dir).glob(f"enainterpolatedsondeC1.c1.{yyyymm}*.nc")))
    if not files:
        print(f"[Skip] No files for {yyyymm}")
        continue

    ds = xr.open_mfdataset(pattern, combine="by_coords")

    ds_vars = ds[vars_to_save]

    ds_filtered = ds_vars.copy()
    ds_filtered["u_wind"] = ds_filtered["u_wind"].where(ds_filtered["qc_u_wind"] == 0)
    ds_filtered["v_wind"] = ds_filtered["v_wind"].where(ds_filtered["qc_v_wind"] == 0)

    ds_filtered = ds_filtered.drop_vars(["qc_u_wind", "qc_v_wind"])
    ds_filtered = ds_filtered.sel(height=target_heights, method="nearest")

    out_path = f"{out_dir}/UV_wind_rose_{yyyymm}.nc"
    ds_filtered.to_netcdf(out_path)
    print(f"[Saved] {out_path}")

    ds.close()


[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201601.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201602.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201603.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201604.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201605.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201606.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201607.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201608.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201609.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201610.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201611.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201612.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_201701.nc
[Saved] /data/ggong/ARM_monthly/ENA/UV_wind_rose/UV_wind_rose_20

In [9]:
ds_2 = xr.open_mfdataset('/data/ggong/ARM_monthly/SGP/UV_wind_rose/*')

In [10]:
START = "2016-01-01 00:00:00"
END   = "2023-12-31 23:59:00"

# 确保时间升序
ds_2 = ds_2.sortby("time")

# 目标 2 分钟时间轴
target_time = pd.date_range(START, END, freq="2min")

# 沿时间维线性插值到 2 分钟分辨率
ds_2min = ds_2.interp(time=target_time, method="linear")

# （可选）检查结果
print(ds_2min)
print(f"新时间点数量: {len(ds_2min.time)}")

<xarray.Dataset> Size: 67MB
Dimensions:  (time: 2103840, height: 3)
Coordinates:
  * height   (height) float32 12B 0.318 1.458 2.458
  * time     (time) datetime64[ns] 17MB 2016-01-01 ... 2023-12-31T23:58:00
Data variables:
    u_wind   (time, height) float32 25MB dask.array<chunksize=(2103840, 3), meta=np.ndarray>
    v_wind   (time, height) float32 25MB dask.array<chunksize=(2103840, 3), meta=np.ndarray>
Attributes: (12/13)
    command_line:          idl -D 0 -R -n interpolatedsonde -s sgp -f C1 -b 2...
    Conventions:           ARM-1.1
    process_version:       vap-interpolatedsonde-6.6-0.el6
    input_datastreams:     sgpgriddedsondeC1.c0 : 2.2 : 20151230.000030-20160...
    dod_version:           interpolatedsonde-c1-4.0
    site_id:               sgp
    ...                    ...
    facility_id:           C1
    data_level:            c1
    location_description:  Southern Great Plains (SGP), Lamont, Oklahoma
    datastream:            sgpinterpolatedsondeC1.c1
    doi:      

In [11]:
ds_2min.to_netcdf('/data/ggong/ARM_monthly/SGP/UV_wind_2min.nc')

## Mete vertical profile

In [15]:
df_top = pd.read_csv('/data/ggong/ARM_monthly/ENA/csv_file/df_top_with_minutes.csv')
df_middle = pd.read_csv('/data/ggong/ARM_monthly/ENA/csv_file/df_top_with_minutes.csv')
df_bottom = pd.read_csv('/data/ggong/ARM_monthly/ENA/csv_file/df_top_with_minutes.csv')

In [25]:
import os, glob
import numpy as np
import pandas as pd
import xarray as xr

sonde_dir = "/data/shared_data/ARM_data/ENA/others/interpolatedsondeC1.c1"
csv_path  = "/data/ggong/ARM_monthly/ENA/csv_file/df_middle_with_minutes.csv"
out_path  = "/data/ggong/ARM_monthly/ENA/mete_profile/middle/ena_sonde_events_qc0_0to8p1km.nc"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

vars_to_save = ["rh","qc_rh","u_wind","qc_u_wind","v_wind","qc_v_wind","temp","qc_temp"]

df = pd.read_csv(csv_path).copy()
df["target_dt"] = pd.to_datetime(df[["year","month","day","hour","minute","second"]])
N = len(df)
time_idx = np.arange(N, dtype=int)

def find_files(yyyymmdd):
    return sorted(glob.glob(os.path.join(sonde_dir, f"enainterpolatedsondeC1.c1.{yyyymmdd}*.nc")))

height_ref = None
H = None
rh_all = T_all = u_all = v_all = None
obs_time = np.full(N, np.datetime64("NaT"), dtype="datetime64[ns]")
tgt_time = df["target_dt"].to_numpy(dtype="datetime64[ns]")

for yyyymmdd, sub in df.groupby(df["target_dt"].dt.strftime("%Y%m%d"), sort=True):
    files = find_files(yyyymmdd)
    if not files:
        continue

    ds = xr.open_mfdataset(files, combine="by_coords")
    hname = "height" if "height" in ds.coords else ("alt" if "alt" in ds.coords else None)
    if hname is None:
        raise ValueError(f"No height coord in {files[0]}")
    hmax = float(ds[hname].max())
    hslice = slice(0.0, 8.1) if hmax <= 50 else slice(0.0, 8100.0)

    dsv = ds.sel({hname: hslice})[vars_to_save]
    rh = dsv["rh"].where(dsv["qc_rh"] == 0)
    T  = dsv["temp"].where(dsv["qc_temp"] == 0)
    u  = dsv["u_wind"].where(dsv["qc_u_wind"] == 0)
    v  = dsv["v_wind"].where(dsv["qc_v_wind"] == 0)

    targets = sub["target_dt"].to_numpy(dtype="datetime64[ns]")
    sel = xr.Dataset({"rh": rh, "temp": T, "u": u, "v": v}).sel(
        time=xr.DataArray(targets, dims="event"), method="nearest"
    )

    if height_ref is None:
        height_ref = sel[hname].values
        H = height_ref.size
        rh_all = np.full((N, H), np.nan, float)
        T_all  = np.full((N, H), np.nan, float)
        u_all  = np.full((N, H), np.nan, float)
        v_all  = np.full((N, H), np.nan, float)

    idx = sub.index.to_numpy(dtype=int)
    obs_time[idx] = sel["time"].values
    rh_all[idx, :] = sel["rh"].values
    T_all[idx, :]  = sel["temp"].values
    u_all[idx, :]  = sel["u"].values
    v_all[idx, :]  = sel["v"].values

ds_out = xr.Dataset(
    data_vars={
        "rh":   (("time","height"), rh_all),
        "temp": (("time","height"), T_all),
        "u":    (("time","height"), u_all),
        "v":    (("time","height"), v_all),
        "obs_time":    (("time",), obs_time),
        "target_time": (("time",), tgt_time),
    },
    coords={"time": time_idx, "height": height_ref},
)

ds_out.to_netcdf(out_path)
print(f"[done] events={ds_out.dims['time']} height={ds_out.dims['height']} -> {out_path}")


[done] events=7317 height=230 -> /data/ggong/ARM_monthly/ENA/mete_profile/middle/ena_sonde_events_qc0_0to8p1km.nc


## Plot mete vertical profile

In [ ]:
ds